In [15]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# Task 1: Synthetic Generative Data Architecture
np.random.seed(42)
mu0 = [2, 2]
mu1 = [5, 5]
mu2 = [2, 5]
sigma = np.array([[1, 0.5], [0.5, 1]])

def generate_data(mu, sigma, n_samples):
    return np.random.multivariate_normal(mu, sigma, n_samples)

X_apples = generate_data(mu0, sigma, 500)
X_oranges = generate_data(mu1, sigma, 500)
X_bananas = generate_data(mu2, sigma, 500)

X = np.vstack((X_apples, X_oranges, X_bananas))
y = np.hstack((np.zeros(500), np.ones(500), 2 * np.ones(500)))

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Apply StandardScaler to the feature
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Task 2: Parameter Estimation
mu0_cap = np.mean(X_train_scaled[y_train == 0], axis=0)
mu1_cap = np.mean(X_train_scaled[y_train == 1], axis=0)
mu2_cap = np.mean(X_train_scaled[y_train == 2], axis=0)

sigmacap = np.cov(np.vstack((X_train_scaled[y_train == 0], X_train_scaled[y_train == 1], X_train_scaled[y_train == 2])), rowvar=False)

phi = np.bincount(y_train).astype(float) / len(y_train)

# Task 3: Model Implementation and Decision Boundaries
def predict_class(x):
    scores = [-(x - mu).T @ np.linalg.inv(sigmacap) @ (x - mu) / 2 + np.log(phi[i]) for i, mu in enumerate([mu0_cap, mu1_cap, mu2_cap])]
    return np.argmax(scores)

y_pred = np.array([predict_class(x) for x in X_test_scaled])

# Boundary visualization
def plot_decision_boundaries(X, y, predict_function):
    h = .02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = np.array([[predict_function(np.array([x1, x2])) for x1 in xx] for x2 in yy])
    
    plt.contourf(xx, yy, Z, alpha=0.8)
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', marker='o')
    for i, mu in enumerate([mu0_cap, mu1_cap, mu2_cap]):
        ellipse = plt.Circle(mu, np.sqrt(np.diag(sigmacap)), color='none', lw=2)
        plt.gca().add_artist(ellipse)
    plt.xlabel('Weight')
    plt.ylabel('Surface Smootheness')
    plt.title('Decision Boundaries and Gaussian Contours')

plt.figure(figsize=(10, 6))
plot_decision_boundaries(X_train_scaled, y_train, predict_class)

# Task 4: Evaluate the Model
conf_matrix = np.zeros((3, 3), dtype=int)
for i in range(len(y_test)):
    conf_matrix[y_test[i], y_pred[i]] += 1

print("Confusion Matrix:")
print(conf_matrix)

TypeError: Cannot cast array data from dtype('float64') to dtype('int64') according to the rule 'safe'